# Fusion Probability Listing

Collect final-selection probabilities from clinical, whole-image radiomics, habitat-sum radiomics, and 3D-DL-to-ML models. The output table has one row per case and one probability feature per modality.

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import json
import numpy as np
import pandas as pd


In [2]:
# ============================================================
# 2. Settings
# ============================================================

TASK = "Prognosis"
LABEL_COL = "Prognosis_label"

model_root = "/host/d/projects/Habitats/models"
task_model_root = os.path.join(model_root, TASK)
fusion_out_dir = os.path.join(task_model_root, "fusion")
os.makedirs(fusion_out_dir, exist_ok=True)

# These are the four final-selection branches to fuse.
# prob_col_name is the final feature name in the fusion table.
modality_configs = [
    {
        "modality": "clinical",
        "model_folder": "clinical",
        "prob_col_name": "prob_clinical",
    },
    {
        "modality": "whole_image",
        "model_folder": "whole_image",
        "prob_col_name": "prob_whole_image",
    },
    {
        "modality": "habitats_avg",
        "model_folder": "habitats_avg",
        "prob_col_name": "prob_habitats_avg",
    },
    {
        "modality": "dl_3d_ml_all",
        "model_folder": "dl_3d_ml_all",
        "prob_col_name": "prob_dl_3d_ml_all",
    },
]

prediction_files = [
    {
        "dataset": "cv",
        "filename": "cv_final_selection_predictions.xlsx",
    },
    {
        "dataset": "internal_test",
        "filename": "internal_test_final_selection_predictions.xlsx",
    },
    {
        "dataset": "external_test",
        "filename": "external_test_final_selection_predictions.xlsx",
    },
]

save_probability_table_path = os.path.join(
    fusion_out_dir,
    "fusion_final_selection_probabilities.xlsx",
)
save_missing_check_path = os.path.join(
    fusion_out_dir,
    "fusion_probability_missing_check.xlsx",
)
save_manifest_path = os.path.join(
    fusion_out_dir,
    "fusion_probability_manifest.json",
)

print("Task model root:", task_model_root)
print("Fusion output dir:", fusion_out_dir)
print("Modalities:")
for cfg in modality_configs:
    print(" ", cfg)


Task model root: /host/d/projects/Habitats/models/Prognosis
Fusion output dir: /host/d/projects/Habitats/models/Prognosis/fusion
Modalities:
  {'modality': 'clinical', 'model_folder': 'clinical', 'prob_col_name': 'prob_clinical'}
  {'modality': 'whole_image', 'model_folder': 'whole_image', 'prob_col_name': 'prob_whole_image'}
  {'modality': 'habitats_avg', 'model_folder': 'habitats_avg', 'prob_col_name': 'prob_habitats_avg'}
  {'modality': 'dl_3d_ml_all', 'model_folder': 'dl_3d_ml_all', 'prob_col_name': 'prob_dl_3d_ml_all'}


In [3]:
# ============================================================
# 3. Helper functions
# ============================================================

KEY_COLS = ["Patient_set", "Patient_index"]
BASE_COLS = ["Patient_set", "Patient_index", LABEL_COL, "split", "fold", "dataset"]


def final_selection_folder(model_folder):
    return os.path.join(task_model_root, model_folder, "final_selections")


def read_one_prediction_file(config, pred_info):
    modality = config["modality"]
    prob_col_name = config["prob_col_name"]
    model_folder = config["model_folder"]
    dataset = pred_info["dataset"]
    filename = pred_info["filename"]

    file_path = os.path.join(final_selection_folder(model_folder), filename)
    if not os.path.isfile(file_path):
        raise FileNotFoundError(file_path)

    df = pd.read_excel(file_path)

    required_cols = KEY_COLS + [LABEL_COL, "prob_final_selection"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if len(missing_cols) > 0:
        raise KeyError(f"Missing columns in {file_path}: {missing_cols}")

    if df.duplicated(KEY_COLS).any():
        dup = df[df.duplicated(KEY_COLS, keep=False)].sort_values(KEY_COLS)
        raise RuntimeError(f"Duplicated patient keys in {file_path}. Example:\n{dup.head()}")

    out = df[KEY_COLS + [LABEL_COL]].copy()
    out["split"] = df["split"] if "split" in df.columns else dataset
    out["fold"] = df["fold"] if "fold" in df.columns else np.nan
    out["dataset"] = dataset
    out[prob_col_name] = df["prob_final_selection"].astype(float)

    # Keep provenance without making these columns part of the feature matrix.
    out[f"{modality}_source_file"] = file_path
    if "final_selection_selected_method" in df.columns:
        out[f"{modality}_final_selection_selected_method"] = df["final_selection_selected_method"].astype(str)

    return out


def read_modality_probabilities(config):
    tables = []
    for pred_info in prediction_files:
        table = read_one_prediction_file(config, pred_info)
        tables.append(table)
        print(
            f"Loaded {config['modality']} {pred_info['dataset']}:",
            table.shape,
        )

    all_df = pd.concat(tables, ignore_index=True)

    if all_df.duplicated(KEY_COLS).any():
        dup = all_df[all_df.duplicated(KEY_COLS, keep=False)].sort_values(KEY_COLS)
        raise RuntimeError(f"Duplicated patient keys after combining {config['modality']}. Example:\n{dup.head()}")

    return all_df


def merge_modality_tables(modality_tables):
    if len(modality_tables) == 0:
        raise RuntimeError("No modality tables provided.")

    merged = modality_tables[0].copy()

    for table in modality_tables[1:]:
        # Check whether shared metadata are consistent before dropping duplicates.
        shared_keys = KEY_COLS
        check_cols = [LABEL_COL, "split", "fold", "dataset"]
        compare = merged[shared_keys + check_cols].merge(
            table[shared_keys + check_cols],
            on=shared_keys,
            how="inner",
            suffixes=("_left", "_right"),
        )
        for col in check_cols:
            left = compare[f"{col}_left"].astype(str).fillna("")
            right = compare[f"{col}_right"].astype(str).fillna("")
            if not (left == right).all():
                mismatch = compare.loc[left != right, shared_keys + [f"{col}_left", f"{col}_right"]]
                print(f"Warning: metadata mismatch for {col}. Showing first rows:")
                display(mismatch.head())

        drop_cols = [col for col in check_cols if col in table.columns]
        table2 = table.drop(columns=drop_cols)
        merged = merged.merge(table2, on=shared_keys, how="outer")

    return merged


def build_missing_check(fusion_df, modality_configs):
    rows = []
    for cfg in modality_configs:
        prob_col = cfg["prob_col_name"]
        rows.append({
            "modality": cfg["modality"],
            "model_folder": cfg["model_folder"],
            "probability_column": prob_col,
            "total_rows": int(fusion_df.shape[0]),
            "available_probability_rows": int(fusion_df[prob_col].notna().sum()),
            "missing_probability_rows": int(fusion_df[prob_col].isna().sum()),
            "missing_patient_keys": fusion_df.loc[fusion_df[prob_col].isna(), KEY_COLS].to_dict("records"),
        })

    dataset_summary = (
        fusion_df
        .groupby("dataset", dropna=False)
        .agg(
            n=("Patient_index", "count"),
            positive_fraction=(LABEL_COL, "mean"),
        )
        .reset_index()
    )

    return pd.DataFrame(rows), dataset_summary


In [4]:
# ============================================================
# 4. Read final-selection probabilities for all modalities
# ============================================================

modality_tables = []
modality_read_summary = []

for cfg in modality_configs:
    print("\n============================================================")
    print("Reading modality:", cfg["modality"])

    df_mod = read_modality_probabilities(cfg)
    modality_tables.append(df_mod)

    modality_read_summary.append({
        "modality": cfg["modality"],
        "model_folder": cfg["model_folder"],
        "probability_column": cfg["prob_col_name"],
        "n_cases": int(df_mod.shape[0]),
        "n_cv": int((df_mod["dataset"] == "cv").sum()),
        "n_internal_test": int((df_mod["dataset"] == "internal_test").sum()),
        "n_external_test": int((df_mod["dataset"] == "external_test").sum()),
        "positive_fraction": float(df_mod[LABEL_COL].mean()),
    })

modality_read_summary_df = pd.DataFrame(modality_read_summary)
display(modality_read_summary_df)



Reading modality: clinical
Loaded clinical cv: (188, 9)
Loaded clinical internal_test: (96, 9)


Loaded clinical external_test: (64, 9)

Reading modality: whole_image
Loaded whole_image cv: (188, 9)
Loaded whole_image internal_test: (96, 9)
Loaded whole_image external_test: (64, 9)

Reading modality: habitats_avg
Loaded habitats_avg cv: (188, 9)
Loaded habitats_avg internal_test: (96, 9)


Loaded habitats_avg external_test: (64, 9)

Reading modality: dl_3d_ml_all
Loaded dl_3d_ml_all cv: (188, 9)
Loaded dl_3d_ml_all internal_test: (96, 9)
Loaded dl_3d_ml_all external_test: (64, 9)


,modality,model_folder,probability_column,n_cases,n_cv,n_internal_test,n_external_test,positive_fraction
0,clinical,clinical,prob_clinical,348,188,96,64,0.258621
1,whole_image,whole_image,prob_whole_image,348,188,96,64,0.258621
2,habitats_avg,habitats_avg,prob_habitats_avg,348,188,96,64,0.258621
3,dl_3d_ml_all,dl_3d_ml_all,prob_dl_3d_ml_all,348,188,96,64,0.258621


In [5]:
# ============================================================
# 5. Merge modalities into one fusion probability table
# ============================================================

fusion_df = merge_modality_tables(modality_tables)

# Put core columns first, then probability features, then provenance columns.
probability_cols = [cfg["prob_col_name"] for cfg in modality_configs]
core_cols = [col for col in BASE_COLS if col in fusion_df.columns]
other_cols = [col for col in fusion_df.columns if col not in core_cols + probability_cols]
fusion_df = fusion_df[core_cols + probability_cols + other_cols]

# Sort for readability.
fusion_df = fusion_df.sort_values(["dataset", "Patient_set", "Patient_index"]).reset_index(drop=True)

print("Fusion table shape:", fusion_df.shape)
print("Probability columns:", probability_cols)
display(fusion_df.head())


,Patient_set,Patient_index,fold_left,fold_right
0,set_1,48,0,3
2,set_1,100,0,1
3,set_1,111,0,3
4,set_1,113,0,4
5,set_1,130,0,3


,Patient_set,Patient_index,fold_left,fold_right
0,set_1,48,0,1
1,set_1,50,0,1
2,set_1,100,0,4
3,set_1,111,0,2
4,set_1,113,0,2


,Patient_set,Patient_index,fold_left,fold_right
0,set_1,48,0,1
1,set_1,50,0,1
2,set_1,100,0,1
3,set_1,111,0,2
4,set_1,113,0,3


Fusion table shape: (348, 18)
Probability columns: ['prob_clinical', 'prob_whole_image', 'prob_habitats_avg', 'prob_dl_3d_ml_all']


,Patient_set,Patient_index,Prognosis_label,split,fold,dataset,prob_clinical,prob_whole_image,prob_habitats_avg,prob_dl_3d_ml_all,clinical_source_file,clinical_final_selection_selected_method,whole_image_source_file,whole_image_final_selection_selected_method,habitats_avg_source_file,habitats_avg_final_selection_selected_method,dl_3d_ml_all_source_file,dl_3d_ml_all_final_selection_selected_method
0,set_1,5,1,train,4,cv,0.406698,0.462886,0.500000,0.654649,/host/d/projects/Habitats/models/Prognosis/cli...,mix,/host/d/projects/Habitats/models/Prognosis/who...,mean,/host/d/projects/Habitats/models/Prognosis/hab...,mix,/host/d/projects/Habitats/models/Prognosis/dl_...,mean
1,set_1,7,1,train,3,cv,0.526426,0.391354,0.373971,0.529406,/host/d/projects/Habitats/models/Prognosis/cli...,mix,/host/d/projects/Habitats/models/Prognosis/who...,mean,/host/d/projects/Habitats/models/Prognosis/hab...,mix,/host/d/projects/Habitats/models/Prognosis/dl_...,mean
2,set_1,8,1,train,2,cv,0.136287,0.332359,0.296098,0.992692,/host/d/projects/Habitats/models/Prognosis/cli...,mix,/host/d/projects/Habitats/models/Prognosis/who...,mean,/host/d/projects/Habitats/models/Prognosis/hab...,mix,/host/d/projects/Habitats/models/Prognosis/dl_...,mean
3,set_1,20,0,train,2,cv,0.136287,0.016032,0.353412,0.255635,/host/d/projects/Habitats/models/Prognosis/cli...,mix,/host/d/projects/Habitats/models/Prognosis/who...,mean,/host/d/projects/Habitats/models/Prognosis/hab...,mix,/host/d/projects/Habitats/models/Prognosis/dl_...,mean
4,set_1,30,0,train,2,cv,0.381038,0.274530,0.041821,0.388681,/host/d/projects/Habitats/models/Prognosis/cli...,mix,/host/d/projects/Habitats/models/Prognosis/who...,mean,/host/d/projects/Habitats/models/Prognosis/hab...,mix,/host/d/projects/Habitats/models/Prognosis/dl_...,mean


In [6]:
# ============================================================
# 6. Missingness and consistency checks
# ============================================================

missing_check_df, dataset_summary_df = build_missing_check(fusion_df, modality_configs)

print("Dataset summary:")
display(dataset_summary_df)

print("Missing probability check:")
display(missing_check_df[[
    "modality",
    "model_folder",
    "probability_column",
    "total_rows",
    "available_probability_rows",
    "missing_probability_rows",
]])

# Hard warning if any modality is missing probabilities.
if (missing_check_df["missing_probability_rows"] > 0).any():
    print("Warning: at least one modality has missing probabilities. See saved missing-check file.")
else:
    print("All modalities have probabilities for every case in the merged table.")


Dataset summary:


,dataset,n,positive_fraction
0,cv,188,0.260638
1,external_test,64,0.265625
2,internal_test,96,0.250000


Missing probability check:


,modality,model_folder,probability_column,total_rows,available_probability_rows,missing_probability_rows
0,clinical,clinical,prob_clinical,348,348,0
1,whole_image,whole_image,prob_whole_image,348,348,0
2,habitats_avg,habitats_avg,prob_habitats_avg,348,348,0
3,dl_3d_ml_all,dl_3d_ml_all,prob_dl_3d_ml_all,348,348,0


All modalities have probabilities for every case in the merged table.


In [7]:
# ============================================================
# 7. Save fusion probability table and manifest
# ============================================================

with pd.ExcelWriter(save_missing_check_path) as writer:
    missing_check_df.to_excel(writer, sheet_name="missing_by_modality", index=False)
    dataset_summary_df.to_excel(writer, sheet_name="dataset_summary", index=False)
    modality_read_summary_df.to_excel(writer, sheet_name="modality_read_summary", index=False)

fusion_df.to_excel(save_probability_table_path, index=False)

manifest = {
    "task": TASK,
    "label_col": LABEL_COL,
    "model_root": model_root,
    "task_model_root": task_model_root,
    "fusion_out_dir": fusion_out_dir,
    "modalities": modality_configs,
    "prediction_files": prediction_files,
    "probability_columns": probability_cols,
    "outputs": {
        "fusion_probability_table": save_probability_table_path,
        "missing_check": save_missing_check_path,
        "manifest": save_manifest_path,
    },
}

with open(save_manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved fusion probability table:", save_probability_table_path)
print("Saved missing check:", save_missing_check_path)
print("Saved manifest:", save_manifest_path)


Saved fusion probability table: /host/d/projects/Habitats/models/Prognosis/fusion/fusion_final_selection_probabilities.xlsx
Saved missing check: /host/d/projects/Habitats/models/Prognosis/fusion/fusion_probability_missing_check.xlsx
Saved manifest: /host/d/projects/Habitats/models/Prognosis/fusion/fusion_probability_manifest.json
